# Phase 3: 探索分析 (EDA)

現在の Parquet データに対して、Phase 2 のバックフィルが進めばそのまま価値が出てくるクエリを実行する。
**結果の信頼性はサンプル数に依存** — 数百レース以下では「方向性の確認」程度にとどめること。

## 主要検証論点

1. **馬番効果**: ばんえいは内外でスタート位置の差が小さいはずだが、実データではどうか
2. **馬場水分量×人気上位的中率**: 市場が水分量変化に追従できているか
3. **減量マーカー出現バリエーション**: ☆以外のマーカー (▲/◇/△) が観察されるか
4. **馬体重増減と着順**: コンディション指標として機能するか
5. **積載重量帯と複勝率**: 重量が増えるほど着順が悪化する想定の検証
6. **人気別単勝回収率**: 市場効率 (穴狙いの優位性が存在するか)

In [ ]:
%cd ..
import pandas as pd
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', 200)

from analytics.eda import (
    open_connection, dataset_summary, date_coverage,
    water_distribution, class_distribution, field_size_distribution,
    post_position_win_rate, popularity_finish_correlation,
    water_band_top3_rate, jockey_allowance_summary,
    body_weight_diff_vs_finish, load_weight_vs_finish, market_efficiency,
)
from analytics.masters import materialize_masters

# マスタを再生成（entries の最新スナップショット反映）
materialize_masters()

con = open_connection()

## 1. データセット規模

In [ ]:
print('=== テーブル別件数 ===')
print(dataset_summary(con))
print()
print('=== 日付カバレッジ ===')
print(date_coverage(con))

## 2. レース属性の分布

In [ ]:
print('--- 馬場水分量 ---')
print(water_distribution(con))
print()
print('--- クラス ---')
print(class_distribution(con))
print()
print('--- 出走頭数 ---')
print(field_size_distribution(con))

## 3. 馬番効果

**仮説**: ばんえいは外枠ほど砂を蹴られにくい、または内ラチ寄りの方が砂が固いなどの偏りがある可能性。
現サンプルでは中枠 (5-7番) が有利傾向だが、信頼区間は広い。

In [ ]:
print(post_position_win_rate(con))

## 4. 人気順別 成績

1番人気の勝率が30%台後半なら市場は素直。50%超だと「堅い」、20%以下なら「荒れる」傾向。

In [ ]:
print(popularity_finish_correlation(con))

## 5. 水分量帯 × 人気上位的中率

市場が馬場状態の変化に追従できていない可能性を探る。極端な馬場(乾/重)では市場が読み誤りやすい想定。

In [ ]:
print(water_band_top3_rate(con))

## 6. 減量マーカー

現状観察マーカー: `☆` のみ (見習い騎手想定)。`▲ / ◇ / △` の出現を Phase 2 バックフィルで収集する。

In [ ]:
print(jockey_allowance_summary(con))

## 7. 馬体重増減 × 着順

「ベスト体重からの乖離」を派生特徴量にする前提の予備分析。

In [ ]:
print(body_weight_diff_vs_finish(con))

## 8. 積載重量 × 着順

ばんえいは積載重量とクラスがほぼ対応 (A1ほど重い)。重量増 → 着順悪化の傾向を確認。

In [ ]:
print(load_weight_vs_finish(con))

## 9. 単勝の市場効率

**回収率 = 払戻金 / 賭け金**。1.0 = 100%回収。胴元控除20-25%があるため通常 0.75〜0.85 程度に収まる。
**1.0を超える人気層がある = 賭けるべき層**。フルバックフィル後の最重要分析。

In [ ]:
print(market_efficiency(con))

## 10. 馬・騎手・調教師マスタ確認

In [ ]:
print('--- 騎手マスタ（出走数順 上位20） ---')
print(con.execute('SELECT * FROM jockeys ORDER BY starts DESC LIMIT 20').fetchdf())
print()
print('--- 調教師マスタ（複勝率順 上位10） ---')
print(con.execute('SELECT * FROM trainers WHERE starts >= 3 ORDER BY top3_rate DESC LIMIT 10').fetchdf())
print()
print('--- 馬マスタ（複数出走馬のみ） ---')
print(con.execute('SELECT * FROM horses WHERE starts >= 2 ORDER BY starts DESC LIMIT 10').fetchdf())

## Phase 4 への観察メモ

現状3日分のデータから、以下は **Phase 4 のクラスタリング軸候補** として有望:

- 馬場水分量 (現サンプルは 2.1〜3.0 の狭いレンジ)
- 1番人気の勝率 (堅い日 vs 荒れる日の判別軸)
- 平均タイムの基準値からの偏差

**フルバックフィル後に再実行すべき分析**:
- 水分量 1.0〜10.0 の全レンジでの 1番人気勝率の変動曲線
- 馬番効果の有意性検定 (二項検定)
- 減量マーカー別の真の勝率差
- 単勝回収率が 1.0 を超える人気層の特定